# Nakamura regular price
## Universidad ICESI 
### David Mauricio Orozco Rios
### author: Davoroz06 - IG

In [ ]:
# libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.stats import gmean
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# options for data display
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text


In [ ]:
regular_window = 7
reference_window = 7
case = "Comparison"

Load data:

In [ ]:
# load retailer data
data = pd.read_csv(wd_dpr + "Filter_Data_{}.csv".format(case))

# Convert 'fecha' to datetime format
data['fecha'] = pd.to_datetime(data['fecha'])

### Nakamura algorithm for sales

0. if $p_t = r_{t-1}$ then $r_t = r_{t-1}$.
1. if $p_t > r_{t-1}$ then $r_t = p_{t}$.
2. if $r_{t-1} \in \{p_{t+1},\dots,p_{t+j}\}$ and the price never rises above $r_{t-1}$ before returning to $r_{t-1}$, then $r_t = r_{t-1}$.
3. If the set $\{p_{t}, p_{t+1}, \dots, p_{t+L}\}$ has K or more different elements, then $r_t = p_t$.
4. Define $p_{max} = max\{p_{t}, p_{t+1}, \dots, p_{t+L}\}$ and $t_{max} = \text{first-time } max\{p_{t}, p_{t+1}, \dots, p_{t+L}\}$. If $p_{max} \in \{p_{tmax+1}, ..., p_{tmax+L}\}$, then $r_{t} = p_{max}$.
5. r_t = p_t.

In the first time period, the algorithm begins at step 3 (the first step that does not refer to a previous regular price).

In [ ]:
def regular_price_nakamura(col_p, L = 1, K = 1, J = regular_window):
    # Regular price list
    r_list = []
    for i in range(len(col_p)):
        if(pd.isna(col_p[i])):
            r_list.append(None)
            continue
        # If we are in the first time period
        if i == 0:
            # Third step
            if len(set(col_p[i:(i+(L+1))])) >= K:
                r = col_p[i]
                r_list.append(r)
                continue
            # Fourth step
            p_max = max(col_p[i:(i+(L+1))])
            t_max = next((j for j, x in enumerate(col_p) if x == p_max), None)
            list_max = col_p[(t_max+1):(t_max+L)]
            if p_max in list_max:
                r = p_max
                r_list.append(r)
                continue
            # fifth step
            r = col_p[i]
            r_list.append(r)
        # For second period onwards
        else:
            # Zero step
            if col_p[i] == r_list[(i-1)]:
                r = r_list[(i-1)]
                r_list.append(r)
                continue
            # First step
            if (r_list[(i-1)] is not None) and (col_p[i] is not None) and (col_p[i] > r_list[(i-1)]):
                r = col_p[i]
                r_list.append(r)
                continue
            # Second step
            list_second = col_p[(i+1):(i+1+J)]
            if (r_list[(i-1)] in list_second):
                reverse_second = next((j for j, x in enumerate(reversed(list_second)) if x == r_list[(i-1)]), None)
                if reverse_second is not None:    
                    t_second = len(list_second) - 1 - reverse_second
                else:
                    t_second = None
                if any(x <= r_list[(i-1)] for x in list_second):
                    r = r_list[(i-1)]
                    r_list.append(r)
                    continue
            # Third step
            if len(set(col_p[i:(i+(L+1))])) >= K:
                r = col_p[i]
                r_list.append(r)
                continue
            # Fourth step
            p_max = max(col_p[i:(i+(L+1))])
            t_max = next((j for j, x in enumerate(col_p) if x == p_max), None)
            list_max = col_p[(t_max+1):(t_max+L)]
            if p_max in list_max:
                r = p_max
                r_list.append(r)
                continue
            # fifth step
            r = col_p[i]
            r_list.append(r)
    return r_list

In [ ]:
# Regular price function
def calculate_regular_price_nakamura(df, l, k, j):
    # Ensure fecha is in datetime format
    df['fecha'] = pd.to_datetime(df['fecha'])
    
    # Sort by tienda, descripcion and fecha to ensure correct time ordering
    df = df.sort_values(by=['tienda', 'descripcion', 'fecha'])
    
    # Apply to a DataFrame
    df['precio_rn'] = df.groupby(['tienda', 'descripcion'])['precio'].transform(lambda x: regular_price_nakamura(x.tolist(), L = l, K = k, J = j))
    
    return df

In [ ]:
nadf = calculate_regular_price_nakamura(data, l = 1, k = 1, j = regular_window)

In [ ]:
nadf.to_csv(wd_dpr + "Data_Regular_Price_{}_{}_{}.csv".format(case, regular_window, reference_window), index = False)